# 03 - Time split, preprocessing, cross-validation and baselines
Rows are sorted by `start.date` (then district); week numbers in the file are unreliable, so the date is used. **Nothing is shuffled.**
Train = all weeks up to and including 2021. Test = 2022 to the end of the data.

In [1]:
# AI-GENERATED (Claude via Antigravity)
import sys, numpy as np, pandas as pd
sys.path.append("../src")
from split import *
from evaluate import regression_scores, to_cases
df = load_features("../data/processed/features.csv")
assert df[DATE].is_monotonic_increasing                     # sorted by time, never shuffled
train, test = time_split(df)
for name, d in [("train", train), ("test", test)]:
    print(f"{name}: {d[DATE].min().date()} to {d[DATE].max().date()} | {len(d):,} rows | {d[DATE].nunique()} weeks")
assert train[DATE].max() < test[DATE].min()                 # test is strictly in the future of train
features = feature_columns(df)
print("number of features:", len(features))
X_train, X_test = train[features], test[features]
y_train_log, y_train_raw, y_test_raw = train[TARGET_LOG], train[TARGET_RAW], test[TARGET_RAW]

train: 2007-02-17 to 2021-12-25 | 19,400 rows | 776 weeks
test: 2022-01-01 to 2025-12-06 | 5,150 rows | 206 weeks
number of features: 51


## Preprocessing (ColumnTransformer)
- `district` -> **one-hot encoding** (one 0/1 column per district). `handle_unknown="ignore"` means a district never seen in training becomes all zeros instead of crashing.
- All other columns pass through unchanged.

**Why no scaling?** Random Forest and boosting models are built from yes/no questions such as "is cases_lag_0 > 12?". Such a question only depends on the *order* of the values, not their size, so rescaling (e.g. rainfall in mm vs. population in millions) changes nothing. Scaling matters for models that mix numbers by distance or weights (k-nearest neighbours, linear models, neural networks). If we add one later, the scaler goes inside the Pipeline so it is fitted on training data only.

In [2]:
# AI-GENERATED (Claude via Antigravity)
pre = make_preprocessor(features)
Z = pre.fit_transform(X_train)                               # fitted on TRAIN only
print("train matrix:", Z.shape, "| test matrix:", pre.transform(X_test).shape)
print("first columns:", list(pre.get_feature_names_out()[:3]), "...")

train matrix: (19400, 75) | test matrix: (5150, 75)
first columns: ['district__district_Ampara', 'district__district_Anuradhapura', 'district__district_Badulla'] ...


## Time-series cross-validation (inside the training set only)
`TimeSeriesSplit(n_splits=5)`: each fold trains on the past and validates on the next block of weeks. I added `gap` = one week (25 rows): the label of the last training week is the first validation week's cases, so that week is skipped to keep the parts apart. The test set is not used here.

In [3]:
# AI-GENERATED (Claude via Antigravity)
cv = make_cv(train)
display(describe_folds(train, cv))                           # also asserts validation is after training

,fold,train from,train to,train rows,valid from,valid to,valid rows
0,1,2007-02-17,2009-08-01,3210,2009-08-08,2012-01-28,3233
1,2,2007-02-17,2012-01-21,6443,2012-01-28,2014-07-26,3233
2,3,2007-02-17,2014-07-19,9676,2014-07-26,2017-01-14,3233
3,4,2007-02-17,2017-01-07,12909,2017-01-14,2019-07-06,3233
4,5,2007-02-17,2019-06-29,16142,2019-07-06,2021-12-25,3233


## Two naive baselines on the test set
- **(a) Persistence:** predict that next week = the most recent week we know (`cases_lag_0`). I read "last week's cases" this way, because the newest known week is the fair naive guess.
- **(b) District average:** each district's average weekly cases over the training set.

In [4]:
# AI-GENERATED (Claude via Antigravity)
district_avg = train.groupby(DISTRICT)[TARGET_RAW].mean()
pred_persistence = test["cases_lag_0"]
pred_avg = test[DISTRICT].map(district_avg)
baselines = pd.DataFrame({"persistence (this week)": regression_scores(y_test_raw, pred_persistence),
                          "district training average": regression_scores(y_test_raw, pred_avg)}).T
display(baselines.round(3))
baselines.to_csv("../reports/baseline_scores.csv")

,MAE,MSE,RMSE,R2,MedAE
persistence (this week),12.802,766.020,27.677,0.805,6.000
district training average,21.906,2397.207,48.961,0.390,8.941


## Why train on log1p(cases)?
Cases are very skewed: many zeros and a few huge outbreaks. With normal training, the biggest weeks dominate the error, so the model chases outbreaks and neglects the many ordinary weeks. `log1p(x) = log(1 + x)` squeezes big numbers (the `+1` keeps zeros valid), so a 10 -> 20 change counts like a 100 -> 200 change. Predictions are turned back with `expm1` before scoring, so all metrics stay in real cases. Below, a deliberately dumb "predict the mean" model shows the effect: trained on log it predicts a *typical* week instead of one pulled up by outbreaks.

**What the result shows (honest reading):** the median error roughly halves (about 22 to 12 cases), so ordinary weeks are predicted much better. But MAE and RMSE get slightly worse, because `expm1` of an average of logs sits below the plain average, so outbreak weeks are under-predicted. Log training is therefore a trade-off: it is good for ordinary weeks and costs some accuracy on outbreaks. We will judge models on MAE, RMSE and MedAE together.

In [5]:
# AI-GENERATED (Claude via Antigravity)
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline
print("skewness  raw: %.2f | log1p: %.2f" % (y_train_raw.skew(), np.log1p(y_train_raw).skew()))
assert np.allclose(np.expm1(np.log1p(y_train_raw)), y_train_raw)       # the round trip is exact
make = lambda: Pipeline([("prep", make_preprocessor(features)), ("model", DummyRegressor(strategy="mean"))])
raw_fit = make().fit(X_train, y_train_raw).predict(X_test)             # trained on raw cases
log_fit = to_cases(make().fit(X_train, y_train_log).predict(X_test))   # trained on log1p, converted back
display(pd.DataFrame({"mean model, raw target": regression_scores(y_test_raw, raw_fit),
                      "mean model, log1p target": regression_scores(y_test_raw, log_fit)}).T.round(3))

skewness  raw: 10.70 | log1p: 0.31


,MAE,MSE,RMSE,R2,MedAE
"mean model, raw target",33.510,4015.937,63.371,-0.021,22.249
"mean model, log1p target",34.421,4928.303,70.202,-0.253,11.683
